# Lesson 05 — HDR Imaging: Merging Exposures

## Why
A standard camera can't capture both deep shadows and bright highlights in a single shot. HDR merges multiple exposures to recover detail from the full dynamic range.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Simulate 3 bracketed exposures from one image
img = cv2.imread('sample.jpg')

dark   = np.clip(img.astype(np.float32) * 0.25, 0, 255).astype(np.uint8)
normal = img.copy()
bright = np.clip(img.astype(np.float32) * 3.5,  0, 255).astype(np.uint8)

print("With real bracketed photos, use:")
print("  images         = [cv2.imread('dark.jpg'), cv2.imread('normal.jpg'), cv2.imread('bright.jpg')]")
print("  exposure_times = np.array([1/100, 1/30, 1/8], dtype=np.float32)")
print()

# Show the three exposures
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, im, t in zip(axes,
    [dark, normal, bright],
    ['Dark (1/100s) — shadow detail', 'Normal (1/30s)', 'Bright (1/8s) — highlight detail']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(t); ax.axis('off')
plt.suptitle('Three exposures — each captures a different part of the dynamic range', fontsize=12)
plt.show()

# Merge using Debevec algorithm (with real photos)
images         = [dark, normal, bright]
exposure_times = np.array([1/100.0, 1/30.0, 1/8.0], dtype=np.float32)

merge_debevec = cv2.createMergeDebevec()
hdr           = merge_debevec.process(images, times=exposure_times)

# Tonemap: compress HDR into 0–255 for display
tonemap = cv2.createTonemap(gamma=2.2)
ldr     = np.clip(tonemap.process(hdr) * 255, 0, 255).astype(np.uint8)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(ldr, cv2.COLOR_BGR2RGB))
plt.title('HDR merged and tonemapped result'); plt.axis('off'); plt.show()

## Key Takeaway
HDR merge recovers shadow detail from the bright exposure and highlight detail from the dark exposure. Tonemapping compresses the wide HDR range back into the limited display range.